# Generative Model Comparison — CVAE-Xception vs CVAE-ViT vs DDPM

**Purpose**: Compare the three conditional generative models on image metrics, physical metrics and qualitative inspection over the **complete test split** (~25,451 conditions; core Part 1 notebook of the article).
**Inputs**: `dataset_unificado_v2.npz`, `modelo_xception_fulldatabaseV3100.h5` (TF, CPU), `ddpm_spines_final_39crop.pt` (PyTorch, cuda:0), `cvae_xception.h5`, `cvae_vit.h5` (after retraining, dataset `weights-cvae-models`), shared `metrics.py` (dataset `physicalmetrics`).
**Outputs**: `image_metrics_per_sample.csv`, `physical_metrics_per_sample.csv` and Figures 1–7 in `OUTPUT_DIR/generative_comparison/` (`.png` + `.svg`).
**Execution environment**: Kaggle / Google Colab (run only one setup cell).
**Dependencies**: tensorflow, torch, transformers, scikit-learn, scikit-image, matplotlib, seaborn, scipy, tqdm, numpy, pandas

In [ ]:
# ⚠ WARNING: CVAE weights (CVAEXPN_PATH, CVAEVIT_PATH) will not be available
# until Notebooks 1 and 2 (cvae_xception_retrain.ipynb, cvae_vit_retrain.ipynb) have
# been run and the resulting checkpoints uploaded to the Kaggle dataset
# carloscanamejoy/weights-cvae-models (as cvae_xception.h5 and cvae_vit.h5).
# All DDPM-only sections can run immediately: every section below detects which
# models are available and silently skips the missing ones.

## Kaggle Environment

In [ ]:
# ============================================================
# ENVIRONMENT SETUP — KAGGLE
# Run this cell only when executing on Kaggle.
# All datasets (including metrics.py from the physicalmetrics
# dataset) are pre-mounted read-only under:
#   /kaggle/input/datasets/carloscanamejoy/
# Nothing to download — paths are defined in the Shared
# Configuration cell below.
# ============================================================
import os, sys
print("Kaggle environment ready:", os.path.exists("/kaggle/input"))

## Google Colab Environment

In [ ]:
# ============================================================
# ENVIRONMENT SETUP — GOOGLE COLAB
# Run this cell only when executing on Google Colab.
# Datasets are downloaded via the Kaggle API using kaggle.json.
# Upload your kaggle.json file to the Colab session before running.
# ============================================================
import os, sys, shutil, zipfile
from google.colab import files

# --- Kaggle credentials ---
os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()                          # upload kaggle.json when prompted
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# --- Install dependencies ---
os.system("pip install -q kaggle torch torchvision tensorflow transformers scikit-learn scikit-image tqdm matplotlib seaborn")

# --- Download datasets ---
os.system("kaggle datasets download -d carloscanamejoy/dataset-spines-united-v2   -p /content/data    --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-xception-model      -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-models              -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/weights-cvae-models         -p /content/weights --unzip")
os.system("kaggle datasets download -d carloscanamejoy/physicalmetrics             -p /content/weights --unzip")

## Load Shared Metrics

In [ ]:
# ── Load shared metrics module from Kaggle dataset ───────────────────────────
# On Kaggle: metrics.py is pre-mounted as part of the physicalmetrics dataset.
# On Colab:  metrics.py is downloaded via the Kaggle API along with the other datasets.

import importlib.util, sys, os

_METRICS_KAGGLE = "/kaggle/input/datasets/carloscanamejoy/physicalmetrics/metrics.py"
_METRICS_COLAB  = "/content/weights/metrics.py"
_METRICS_LOCAL  = os.path.join("..", "utils", "metrics.py")    # running from the repo
_metrics_path   = next(p for p in (_METRICS_KAGGLE, _METRICS_COLAB, _METRICS_LOCAL)
                       if os.path.exists(p))

spec    = importlib.util.spec_from_file_location("metrics", _metrics_path)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules["metrics"] = metrics

from metrics import (
    STRUCTURE_MAP, STRUCTURE_NAMES, STRUCTURE_COLORS, MODEL_COLORS, PARAM_NAMES,
    circular_mask, MASK,
    masked_mse, masked_bce, masked_ssim,
    cosine_similarity_pair, cosine_similarity_batch,
    magnetization, abs_magnetization, cnn_correlation,
    structure_factor, azimuthal_average, oz_fit, chi_ensemble,
    shift_image, reflect_image,
    normalize_metrics,
    save_figure, apply_figure_style,
    center_crop, get_structure_label,
)
print(f"metrics module loaded from: {_metrics_path}")

## Shared Configuration

Auto-detects the execution environment (`_ON_KAGGLE`) and defines every path used
by this notebook — the rest of the code never hardcodes paths.

In [ ]:
import math
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import pearsonr

SEED = 42

# --- Environment auto-detection: single source of truth for all paths ---
_ON_KAGGLE = os.path.exists("/kaggle/input")

DATASET_PATH  = ("/kaggle/input/datasets/carloscanamejoy/dataset-spines-united-v2/dataset_unificado_v2.npz"
                 if _ON_KAGGLE else "/content/data/dataset_unificado_v2.npz")
XCEPTION_PATH = ("/kaggle/input/datasets/carloscanamejoy/weights-xception-model/modelo_xception_fulldatabaseV3100.h5"
                 if _ON_KAGGLE else "/content/weights/modelo_xception_fulldatabaseV3100.h5")
DDPM_PATH     = ("/kaggle/input/datasets/carloscanamejoy/weights-models/ddpm_spines_final_39crop.pt"
                 if _ON_KAGGLE else "/content/weights/ddpm_spines_final_39crop.pt")
CVAEXPN_PATH  = ("/kaggle/input/datasets/carloscanamejoy/weights-cvae-models/cvae_xception.h5"
                 if _ON_KAGGLE else "/content/weights/cvae_xception.h5")
CVAEVIT_PATH  = ("/kaggle/input/datasets/carloscanamejoy/weights-cvae-models/cvae_vit.h5"
                 if _ON_KAGGLE else "/content/weights/cvae_vit.h5")
METRICS_PATH  = ("/kaggle/input/datasets/carloscanamejoy/physicalmetrics/metrics.py"
                 if _ON_KAGGLE else "/content/weights/metrics.py")
OUTPUT_DIR    = "/kaggle/working/outputs" if _ON_KAGGLE else "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DDPM conditioning scaler (saved next to the checkpoint at training time)
DDPM_SCALER_PKL = os.path.join(os.path.dirname(DDPM_PATH),
                               "param_scaler_ddpm_final_39crop.pkl")

apply_figure_style()

K = 32               # ensemble size per parameter vector θ
FAST_SAMPLING_STEPS = 100
OZ_R2_VALID = 0.7    # OZ fits below this R² are flagged as invalid

FIG_DIR = os.path.join(OUTPUT_DIR, "generative_comparison")
os.makedirs(FIG_DIR, exist_ok=True)
MODEL_KEYS  = ["ddpm", "cvae_xcp", "cvae_vit"]
MODEL_LABEL = {"ddpm": "DDPM", "cvae_xcp": "CVAE-Xception", "cvae_vit": "CVAE-ViT"}
N_STRUCT = len(STRUCTURE_NAMES)

## Section 0 — Shared Setup

### 0.1 Dataset and complete test split

70/15/15 split, `SEED=42`, stratified by cluster label. The evaluation runs on the
**complete test split** — no further subsampling.

In [ ]:
data     = np.load(DATASET_PATH, mmap_mode="r")
print(f"npz keys: {data.files}")
imgs     = data["img"]       # (N, 39, 39, 1)
params   = np.asarray(data["params"]).astype(np.float32)
clusters = np.asarray(data["labels"]).astype(int)
all_idx  = np.arange(len(imgs))
N = len(imgs)

idx_train, idx_temp = train_test_split(all_idx, test_size=0.30, random_state=SEED,
                                       stratify=clusters)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED,
                                     stratify=clusters[idx_temp])

# COMPLETE test split — no further subsampling
eval_idx = np.sort(idx_test)
n_eval   = len(eval_idx)

ref_imgs = np.asarray(imgs[eval_idx], dtype=np.float32)
if ref_imgs.ndim == 4:
    ref_imgs = ref_imgs[..., 0]                              # (N_test, 39, 39)
ref_params = params[eval_idx]                                # (N_test, 8)
ref_struct = np.array([get_structure_label(c) for c in clusters[eval_idx]])

print(f"Test split size: {n_eval} samples")
print({s: int((ref_struct == s).sum()) for s in STRUCTURE_NAMES})

### 0.2 TF + PyTorch coexistence and Xception inverse model (CPU)

In [ ]:
# TensorFlow: allow GPU memory growth (prevents TF from claiming all VRAM)
import tensorflow as tf
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# PyTorch: pin explicitly to cuda:0 to avoid multi-GPU device mismatch
import torch
import torch.nn as nn
import torch.nn.functional as F
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")


def build_xception(n_out=8):
    inp  = tf.keras.Input(shape=(224, 224, 3))
    base = tf.keras.applications.Xception(weights=None, include_top=False, input_tensor=inp)
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.BatchNormalization(name="batch_normalization_4")(x)
    x = tf.keras.layers.Dropout(0.4, name="dropout")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="dense")(x)
    x = tf.keras.layers.BatchNormalization(name="batch_normalization_5")(x)
    x = tf.keras.layers.Dropout(0.3, name="dropout_1")(x)
    return tf.keras.Model(inp, tf.keras.layers.Dense(n_out, activation="linear", name="dense_1")(x))


# Always load on CPU to avoid contention with the PyTorch DDPM on GPU
with tf.device("/cpu:0"):
    xception_model = build_xception()
    xception_model.load_weights(XCEPTION_PATH)
feature_extractor = tf.keras.Model(inputs=xception_model.input,
                                   outputs=xception_model.get_layer("dense").output)
print(f"Xception loaded on CPU — {xception_model.count_params():,} parameters")


def extract_xception_features(imgs_batch, batch_size=32, chunk=2048):
    """256-dim Dense-layer features for (B, 39, 39) images in [-1, 1], chunked."""
    out = np.empty((len(imgs_batch), 256), dtype=np.float32)
    for i0 in tqdm(range(0, len(imgs_batch), chunk), desc="features", leave=False):
        i1 = min(i0 + chunk, len(imgs_batch))
        x = tf.image.resize(np.asarray(imgs_batch[i0:i1], dtype=np.float32)[..., np.newaxis],
                            (224, 224)).numpy()
        x = np.repeat(x, 3, axis=-1)
        x = (x + 1) / 2.0
        x = tf.keras.applications.xception.preprocess_input(x * 255.0)
        with tf.device("/cpu:0"):
            out[i0:i1] = feature_extractor.predict(x, batch_size=batch_size, verbose=0)
    return out

### 0.3 DDPM (PyTorch, cuda:0)

Architecture and fast sampler are identical to `ddpm_spines_train.ipynb`.
The model was trained on 40×40 reflect-padded images and conditioned on
MinMax-normalized θ; generated images are cropped back to 39×39.

In [ ]:
def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half - 1))
    args = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)


class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))

    def forward(self, t, cond):
        t_emb = sinusoidal_embedding(t, self.t_mlp[0].in_features)
        return self.t_mlp(t_emb) + self.c_mlp(cond)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = F.silu(self.norm2(h))
        h = self.dropout(h)
        h = self.conv2(h)
        return h + self.skip(x)


class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, ch)
        self.qkv  = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        q, k, v = self.qkv(h).chunk(3, dim=1)
        q = q.reshape(B, C, -1); k = k.reshape(B, C, -1); v = v.reshape(B, C, -1)
        attn = torch.softmax(torch.bmm(q.transpose(1, 2), k) / math.sqrt(C), dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2)).reshape(B, C, H, W)
        return x + self.proj(out)


class ConditionalUNet(nn.Module):
    """Conditional U-Net for the DDPM (same as in ddpm_spines_train.ipynb)."""

    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1, 2, 4),
                 cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        chs = [base_ch * m for m in ch_mults]
        self.emb = TimeCondEmbedding(t_dim=emb_dim, cond_in=cond_dim, out_dim=emb_dim)
        self.conv_in = nn.Conv2d(img_channels, chs[0], 3, padding=1)

        self.down_blocks, self.down_samples = nn.ModuleList(), nn.ModuleList()
        in_ch = chs[0]
        self.skip_channels = []
        for i, out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([
                ResBlock(in_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout)]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)
                                     if i < len(chs) - 1 else nn.Identity())
            in_ch = out_ch

        self.mid_block1 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.mid_attn   = SelfAttention(chs[-1])
        self.mid_block2 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)

        self.up_blocks, self.up_samples = nn.ModuleList(), nn.ModuleList()
        for i, out_ch in enumerate(reversed(chs)):
            skip_ch = self.skip_channels[-(i + 1)]
            self.up_blocks.append(nn.ModuleList([
                ResBlock(in_ch + skip_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout)]))
            self.up_samples.append(nn.ConvTranspose2d(out_ch, out_ch, 4, stride=2, padding=1)
                                   if i < len(chs) - 1 else nn.Identity())
            in_ch = out_ch

        self.norm_out = nn.GroupNorm(8, chs[0])
        self.conv_out = nn.Conv2d(chs[0], img_channels, 1)

    def forward(self, x, t, cond):
        emb = self.emb(t, cond)
        h = self.conv_in(x)
        skips = []
        for (rb1, rb2), ds in zip(self.down_blocks, self.down_samples):
            h = rb1(h, emb); h = rb2(h, emb)
            skips.append(h)
            h = ds(h)
        h = self.mid_block1(h, emb); h = self.mid_attn(h); h = self.mid_block2(h, emb)
        for (rb1, rb2), us, skip in zip(self.up_blocks, self.up_samples, reversed(skips)):
            h = torch.cat([h, skip], dim=1)
            h = rb1(h, emb); h = rb2(h, emb)
            h = us(h)
        h = F.silu(self.norm_out(h))
        return self.conv_out(h)


class DDPMScheduler:
    """Beta schedule for the DDPM ('linear' or 'cosine')."""

    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, schedule="cosine", device="cpu"):
        self.T = T
        if schedule == "linear":
            betas = torch.linspace(beta_start, beta_end, T, device=device)
        elif schedule == "cosine":
            steps, s = T + 1, 0.008
            x = torch.linspace(0, T, steps, device=device)
            ac = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
            ac = ac / ac[0]
            betas = (1 - (ac[1:] / ac[:-1])).clamp(max=0.999)
        else:
            raise ValueError(f"Unknown schedule: {schedule}")
        alphas = 1.0 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.sqrt_alphas_cumprod = ac.sqrt()
        self.sqrt_one_minus_alphas_cumprod = (1.0 - ac).sqrt()


@torch.no_grad()
def fast_sample(model, cond, scheduler, n_steps=100, img_size=40):
    """DDIM-style deterministic-noise fast sampler (same as in training notebook)."""
    B = cond.shape[0]
    x = torch.randn(B, 1, img_size, img_size, device=cond.device)
    timesteps = list(range(0, scheduler.T, scheduler.T // n_steps))[::-1]
    for t_val in timesteps:
        t_tensor = torch.full((B,), t_val, device=cond.device, dtype=torch.long)
        eps_pred = model(x, t_tensor, cond)
        sqrt_a  = scheduler.sqrt_alphas_cumprod[t_val]
        sqrt_1a = scheduler.sqrt_one_minus_alphas_cumprod[t_val]
        x0_pred = ((x - sqrt_1a * eps_pred) / sqrt_a).clamp(-1, 1)
        if t_val > 0:
            t_prev = max(t_val - scheduler.T // n_steps, 0)
            x = (scheduler.sqrt_alphas_cumprod[t_prev] * x0_pred
                 + scheduler.sqrt_one_minus_alphas_cumprod[t_prev] * eps_pred)
        else:
            x = x0_pred
    return x

In [ ]:
ckpt = torch.load(DDPM_PATH, map_location=DEVICE, weights_only=False)
hp = ckpt["hyperparams"]
ddpm = ConditionalUNet(img_channels=1, base_ch=hp["base_ch"], ch_mults=(1, 2, 4),
                       cond_dim=8, emb_dim=hp["cond_emb_dim"], dropout=0.0).to(DEVICE)
state = ckpt["ema"] if ckpt.get("ema") is not None else ckpt["model"]
ddpm.load_state_dict(state)
ddpm.eval()
MODEL_DEVICE = next(ddpm.parameters()).device
ddpm_scheduler = DDPMScheduler(T=1000, schedule=hp["beta_schedule"], device=MODEL_DEVICE)
print(f"DDPM loaded on {MODEL_DEVICE} (EMA weights: {ckpt.get('ema') is not None})")


def rebuild_ddpm_scaler(params_full, seed=42):
    """Replicates make_split() from ddpm_spines_train.ipynb to recover the
    MinMaxScaler the DDPM was conditioned with (fit on its own training split)."""
    n = len(params_full)
    sub_idx = np.random.RandomState(seed).choice(n, size=n, replace=False)
    p_s = params_full[sub_idx]
    idx = np.arange(n)
    idx_tr, _ = train_test_split(idx, test_size=0.30, random_state=seed)
    return MinMaxScaler().fit(p_s[idx_tr])


if os.path.exists(DDPM_SCALER_PKL):
    with open(DDPM_SCALER_PKL, "rb") as f:
        scaler_ddpm = pickle.load(f)
    print("DDPM parameter scaler loaded from pickle")
else:
    scaler_ddpm = rebuild_ddpm_scaler(params)
    print("DDPM parameter scaler rebuilt from the training split recipe")

### 0.4 CVAE models (TensorFlow)

The class definitions below mirror `cvae_xception_retrain.ipynb` and
`cvae_vit_retrain.ipynb` exactly, so `load_weights` restores the trained models.
Both CVAEs were conditioned on θ normalized with a MinMaxScaler fitted on the
stratified training split (rebuilt deterministically here).

In [ ]:
from tensorflow.keras import layers

scaler_cvae = MinMaxScaler().fit(params[idx_train])


def _res_block_tf(x, ch):
    h = layers.Conv2D(ch, 3, padding="same", activation="relu")(x)
    h = layers.Conv2D(ch, 3, padding="same")(h)
    return layers.Activation("relu")(layers.Add()([x, h]))


def build_cvae_decoder(z_dim, dec_emb_dim):
    z    = layers.Input(shape=(z_dim,), name="z")
    cond = layers.Input(shape=(8,), name="theta_norm")
    c = layers.Dense(dec_emb_dim, activation="relu", name="dec_cond_emb")(cond)
    h = layers.Concatenate()([z, c])
    h = layers.Dense(5 * 5 * 256, activation="relu")(h)
    h = layers.Reshape((5, 5, 256))(h)
    h = layers.Conv2DTranspose(128, 4, strides=2, padding="same", activation="relu")(h)
    h = layers.Conv2DTranspose(64, 4, strides=2, padding="same", activation="relu")(h)
    h = layers.Conv2DTranspose(32, 4, strides=2, padding="same", activation="relu")(h)
    h = _res_block_tf(h, 32)
    h = _res_block_tf(h, 32)
    h = layers.Conv2D(1, 3, padding="same", activation="tanh")(h)
    out = layers.Cropping2D(((0, 1), (0, 1)), name="crop_39")(h)
    return tf.keras.Model([z, cond], out, name="decoder")


def build_cvae_posterior(feat_dim, z_dim, cond_emb_dim):
    feat = layers.Input(shape=(feat_dim,), name="features")
    cond = layers.Input(shape=(8,), name="theta_norm")
    c = layers.Dense(cond_emb_dim, activation="relu", name="post_cond_emb")(cond)
    h = layers.Concatenate()([feat, c])
    h = layers.Dense(512, activation="relu")(h)
    h = layers.Dense(256, activation="relu")(h)
    mu      = layers.Dense(z_dim, name="mu_q")(h)
    log_var = layers.Dense(z_dim, name="log_var_q")(h)
    return tf.keras.Model([feat, cond], [mu, log_var], name="posterior_q")


def build_cvae_prior(z_dim, cond_emb_dim):
    cond = layers.Input(shape=(8,), name="theta_norm")
    c = layers.Dense(cond_emb_dim, activation="relu", name="prior_cond_emb")(cond)
    h = layers.Dense(64, activation="relu")(c)
    h = layers.Dense(128, activation="relu")(h)
    mu      = layers.Dense(z_dim, name="mu_p")(h)
    log_var = layers.Dense(z_dim, name="log_var_p")(h)
    return tf.keras.Model(cond, [mu, log_var], name="prior_p")


def build_xcp_backbone():
    inp = layers.Input(shape=(39, 39, 1), name="image_39")
    x = layers.Resizing(224, 224, interpolation="bilinear")(inp)
    x = layers.Concatenate()([x, x, x])
    base = tf.keras.applications.Xception(weights=None, include_top=False, input_tensor=x)
    feat = layers.GlobalAveragePooling2D(name="gap")(base.output)
    return tf.keras.Model(inp, feat, name="xception_backbone")


class CVAEXception(tf.keras.Model):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.backbone  = build_xcp_backbone()
        self.posterior = build_cvae_posterior(2048, 128, 32)
        self.prior     = build_cvae_prior(128, 32)
        self.decoder   = build_cvae_decoder(128, 64)
        self.beta = tf.Variable(1e-6, trainable=False, dtype=tf.float32, name="beta")

    def call(self, inputs, training=False):
        x, y = inputs
        feat = self.backbone(x, training=training)
        mu_q, log_var_q = self.posterior([feat, y], training=training)
        z = mu_q + tf.exp(0.5 * log_var_q) * tf.random.normal(tf.shape(mu_q))
        return self.decoder([z, y], training=training)

    def sample_from_prior(self, y_norm, n_samples=1):
        y = tf.repeat(tf.convert_to_tensor(y_norm, tf.float32), n_samples, axis=0)
        mu_p, log_var_p = self.prior(y, training=False)
        z = mu_p + tf.exp(0.5 * log_var_p) * tf.random.normal(tf.shape(mu_p))
        x = self.decoder([z, y], training=False)
        return tf.reshape(x, (len(y_norm), n_samples, 39, 39)).numpy()


class CVAEViTComparison(tf.keras.Model):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        from transformers import TFViTModel

        class ViTBackbone(tf.keras.layers.Layer):
            def __init__(self, **kw):
                super().__init__(**kw)
                self.vit = TFViTModel.from_pretrained("google/vit-base-patch16-224-in21k")

            def call(self, x, training=False):
                x = tf.image.resize(x, (224, 224))
                x = tf.image.grayscale_to_rgb(x)
                x = tf.transpose(x, [0, 3, 1, 2])
                out = self.vit(pixel_values=x, training=training).last_hidden_state
                return tf.reduce_mean(out, axis=1)

        self.backbone  = ViTBackbone(name="vit_backbone")
        self.posterior = build_cvae_posterior(768, 128, 64)
        self.prior     = build_cvae_prior(128, 64)
        self.decoder   = build_cvae_decoder(128, 64)
        self.beta = tf.Variable(1e-6, trainable=False, dtype=tf.float32, name="beta")

    def call(self, inputs, training=False):
        x, y = inputs
        feat = self.backbone(x, training=training)
        mu_q, log_var_q = self.posterior([feat, y], training=training)
        z = mu_q + tf.exp(0.5 * log_var_q) * tf.random.normal(tf.shape(mu_q))
        return self.decoder([z, y], training=training)

    def sample_from_prior(self, y_norm, n_samples=1):
        y = tf.repeat(tf.convert_to_tensor(y_norm, tf.float32), n_samples, axis=0)
        mu_p, log_var_p = self.prior(y, training=False)
        z = mu_p + tf.exp(0.5 * log_var_p) * tf.random.normal(tf.shape(mu_p))
        x = self.decoder([z, y], training=False)
        return tf.reshape(x, (len(y_norm), n_samples, 39, 39)).numpy()

In [ ]:
AVAILABLE = {"ddpm": True}
_dummy = (ref_imgs[:2, :, :, None], scaler_cvae.transform(ref_params[:2]).astype(np.float32))

cvae_xcp = None
if os.path.exists(CVAEXPN_PATH):
    cvae_xcp = CVAEXception(name="cvae_xception")
    _ = cvae_xcp(_dummy)
    cvae_xcp.load_weights(CVAEXPN_PATH)
    AVAILABLE["cvae_xcp"] = True
    print("CVAE-Xception weights loaded")
else:
    AVAILABLE["cvae_xcp"] = False
    print(f"⚠ CVAE-Xception weights not found at {CVAEXPN_PATH} — model skipped")

cvae_vit = None
if os.path.exists(CVAEVIT_PATH):
    cvae_vit = CVAEViTComparison(name="cvae_vit")
    _ = cvae_vit(_dummy)
    cvae_vit.load_weights(CVAEVIT_PATH)
    AVAILABLE["cvae_vit"] = True
    print("CVAE-ViT weights loaded")
else:
    AVAILABLE["cvae_vit"] = False
    print(f"⚠ CVAE-ViT weights not found at {CVAEVIT_PATH} — model skipped")

ACTIVE_MODELS = [m for m in MODEL_KEYS if AVAILABLE[m]]
print(f"Active models: {[MODEL_LABEL[m] for m in ACTIVE_MODELS]}")

## Section 1 — Image Generation (K = 32 per θ)

For every test-split parameter vector θ each model generates an ensemble of
K = 32 images. The DDPM uses the 100-step fast sampler at 40×40 and **crops to
39×39 (top-left, no interpolation) before any downstream use**; the CVAEs sample
z ~ p(z|θ) and decode (their decoder already crops 40→39 internally).

Ensembles are written incrementally to memory-mapped `.npy` caches with JSON
progress markers, so an interrupted run resumes where it stopped and re-running
the notebook skips this (expensive) section entirely.

In [ ]:
import json

THETA_PER_CALL = 8                       # DDPM: 8 θ × K=32 → batches of 256 images
CVAE_CHUNK     = 64

cond_norm_ddpm = scaler_ddpm.transform(ref_params).astype(np.float32)
cond_norm_cvae = scaler_cvae.transform(ref_params).astype(np.float32)


def fill_ddpm(samples, i0, i1):
    cond = torch.from_numpy(np.repeat(cond_norm_ddpm[i0:i1], K, axis=0)).to(MODEL_DEVICE)
    x = fast_sample(ddpm, cond, ddpm_scheduler, n_steps=FAST_SAMPLING_STEPS, img_size=40)
    # DDPM generates (B, 40, 40) — crop to (B, 39, 39) before any downstream use
    x39 = x[:, 0, :39, :39].cpu().numpy().astype(np.float16)   # top-left crop, no interpolation
    samples[i0:i1] = x39.reshape(i1 - i0, K, 39, 39)


def fill_cvae(model):
    def _fill(samples, i0, i1):
        samples[i0:i1] = model.sample_from_prior(
            cond_norm_cvae[i0:i1], n_samples=K).astype(np.float16)
    return _fill


GENERATORS = {"ddpm": (fill_ddpm, THETA_PER_CALL),
              "cvae_xcp": (fill_cvae(cvae_xcp), CVAE_CHUNK),
              "cvae_vit": (fill_cvae(cvae_vit), CVAE_CHUNK)}

ensembles = {}
for m in ACTIVE_MODELS:
    cache    = os.path.join(OUTPUT_DIR, f"gen_cache_{m}_N{n_eval}_K{K}.npy")
    progress = cache + ".progress.json"
    if os.path.exists(cache) and os.path.exists(progress):
        samples = np.lib.format.open_memmap(cache, mode="r+")
        with open(progress) as f:
            start = json.load(f)["done"]
    else:
        samples = np.lib.format.open_memmap(cache, mode="w+", dtype=np.float16,
                                            shape=(n_eval, K, 39, 39))
        start = 0
    fill_fn, step = GENERATORS[m]
    if start < n_eval:
        t0 = time.time()
        for i0 in tqdm(range(start, n_eval, step), desc=f"generate {MODEL_LABEL[m]}"):
            i1 = min(i0 + step, n_eval)
            fill_fn(samples, i0, i1)
            if (i0 // step) % 50 == 0:
                samples.flush()
                with open(progress, "w") as f:
                    json.dump({"done": i1}, f)
        samples.flush()
        with open(progress, "w") as f:
            json.dump({"done": n_eval}, f)
        print(f"{MODEL_LABEL[m]}: generated {samples.shape} in {(time.time()-t0)/60:.1f} min")
    else:
        print(f"{MODEL_LABEL[m]}: loaded cached ensembles {samples.shape}")
    ensembles[m] = samples

In [ ]:
# Best sample per θ = highest masked SSIM vs the reference image (Regime A)
best_idx, best_samples = {}, {}
for m in ACTIVE_MODELS:
    bcache = os.path.join(OUTPUT_DIR, f"best_idx_{m}_N{n_eval}_K{K}.npy")
    if os.path.exists(bcache):
        bi = np.load(bcache)
    else:
        bi = np.empty(n_eval, dtype=int)
        for i in tqdm(range(n_eval), desc=f"best sample {MODEL_LABEL[m]}", leave=False):
            ref = ref_imgs[i]
            ens_i = ensembles[m][i].astype(np.float32)
            scores = [masked_ssim(ref, ens_i[j]) for j in range(K)]
            bi[i] = int(np.argmax(scores))
        np.save(bcache, bi)
    best_idx[m] = bi
    best_samples[m] = np.stack([ensembles[m][i, bi[i]].astype(np.float32)
                                for i in range(n_eval)])
    print(f"{MODEL_LABEL[m]}: best samples selected {best_samples[m].shape}")

## Section 2 — Image Metrics (Regime A — single best image per θ)

Masked MSE / BCE / SSIM and Xception-latent cosine similarity between each
reference and the model's best sample.

In [ ]:
df_img = pd.DataFrame({"idx": eval_idx, "structure": ref_struct})
for p, name in enumerate(["T", "J2", "KDM", "Hex", "KanS", "Kan1", "J3", "J4"]):
    df_img[f"theta_{name}"] = ref_params[:, p]

# Reference features (cached: extraction over the full split is expensive on CPU)
ZREF_CACHE = os.path.join(OUTPUT_DIR, f"zref_N{n_eval}.npy")
if os.path.exists(ZREF_CACHE):
    z_ref = np.load(ZREF_CACHE)
else:
    z_ref = extract_xception_features(ref_imgs)
    np.save(ZREF_CACHE, z_ref)

for m in ACTIVE_MODELS:
    gen = best_samples[m]
    df_img[f"{m}_mse"]  = [masked_mse(ref_imgs[i], gen[i]) for i in range(n_eval)]
    df_img[f"{m}_bce"]  = [masked_bce(ref_imgs[i], gen[i]) for i in range(n_eval)]
    df_img[f"{m}_ssim"] = [masked_ssim(ref_imgs[i], gen[i])
                           for i in tqdm(range(n_eval), desc=f"SSIM {MODEL_LABEL[m]}",
                                         leave=False)]
    zg_cache = os.path.join(OUTPUT_DIR, f"zgen_{m}_N{n_eval}.npy")
    if os.path.exists(zg_cache):
        z_gen = np.load(zg_cache)
    else:
        z_gen = extract_xception_features(gen)
        np.save(zg_cache, z_gen)
    df_img[f"{m}_cosine"] = cosine_similarity_batch(z_ref, z_gen)

df_img.to_csv(os.path.join(FIG_DIR, "image_metrics_per_sample.csv"), index=False)

print("Overall image metrics (mean ± std):")
for m in ACTIVE_MODELS:
    row = " | ".join(f"{k}: {df_img[f'{m}_{k}'].mean():.4f} ± {df_img[f'{m}_{k}'].std():.4f}"
                     for k in ["mse", "bce", "ssim", "cosine"])
    print(f"  {MODEL_LABEL[m]:>14}: {row}")

print("\nPer-structure SSIM / cosine:")
print(df_img.groupby("structure")[[f"{m}_{k}" for m in ACTIVE_MODELS
                                   for k in ("ssim", "cosine")]].mean().round(4))

## Section 3 — Physical Metrics

- **Regime A** — single best image per θ: magnetization M, nearest-neighbor
  correlation C_nn, and the Ornstein-Zernike proxies χ and ξ (with fit R²).
- **Regime B** — K-sample ensemble: ensemble means of M and C_nn, and the
  fluctuation-dissipation susceptibility χ_ens = N·Var(m)/T.

> The OZ fit is physically meaningful only in disordered / high-temperature
> regimes; in ordered phases (Ferromagnetic, Helical, Skyrmions, …) the fit R²
> is low. R² is always reported alongside χ and ξ, and low-R² fits are flagged
> in the figures.

In [ ]:
T_FLOOR = 1e-3                       # avoid division by ~0 temperatures in χ_ens
temps = np.maximum(ref_params[:, 0], T_FLOOR)

phys = {"structure": ref_struct, "idx": eval_idx}

# Reference (original) physical quantities
phys["orig_M"]   = np.array([magnetization(im) for im in ref_imgs])
phys["orig_Cnn"] = np.array([cnn_correlation(im)
                             for im in tqdm(ref_imgs, desc="Cnn originals")])
oz_orig = np.array([oz_fit(im) for im in tqdm(ref_imgs, desc="OZ fit originals")])
phys["orig_chi_proxy"], phys["orig_xi"], phys["orig_oz_r2"] = oz_orig.T

for m in ACTIVE_MODELS:
    gen = best_samples[m]
    # Regime A — best single image
    phys[f"{m}_M_A"]   = np.array([magnetization(im) for im in gen])
    phys[f"{m}_Cnn_A"] = np.array([cnn_correlation(im)
                                   for im in tqdm(gen, desc=f"Cnn_A {MODEL_LABEL[m]}",
                                                  leave=False)])
    oz_gen = np.array([oz_fit(im) for im in tqdm(gen, desc=f"OZ fit {MODEL_LABEL[m]}")])
    phys[f"{m}_chi_proxy_A"], phys[f"{m}_xi_A"], phys[f"{m}_oz_r2_A"] = oz_gen.T
    # Regime B — ensemble statistics (memmap slices: one K-ensemble in RAM at a time)
    M_B, Cnn_B, chi_B = (np.empty(n_eval) for _ in range(3))
    for i in tqdm(range(n_eval), desc=f"ensemble physics {MODEL_LABEL[m]}"):
        ens_i = ensembles[m][i].astype(np.float32)
        M_B[i]   = np.mean([magnetization(im) for im in ens_i])
        Cnn_B[i] = np.mean([cnn_correlation(im) for im in ens_i])
        chi_B[i] = chi_ensemble(ens_i, temperature=temps[i])
    phys[f"{m}_M_B"], phys[f"{m}_Cnn_B"], phys[f"{m}_chi_ens_B"] = M_B, Cnn_B, chi_B

df_phys = pd.DataFrame(phys)
# The three χ comparisons per model: ① orig proxy vs gen proxy,
# ② gen proxy vs gen ensemble, ③ orig proxy vs gen ensemble
for m in ACTIVE_MODELS:
    df_phys[f"{m}_chi_orig_proxy"] = df_phys["orig_chi_proxy"]
    df_phys[f"{m}_chi_gen_proxy"]  = df_phys[f"{m}_chi_proxy_A"]
    df_phys[f"{m}_chi_gen_ens"]    = df_phys[f"{m}_chi_ens_B"]

df_phys.to_csv(os.path.join(FIG_DIR, "physical_metrics_per_sample.csv"), index=False)
print(df_phys.filter(regex="_M_|_Cnn_|chi_ens").describe().round(4))

## Section 4 — Figures

**Figure 1** — overall image metrics per model.

In [ ]:
IMG_METRICS = ["mse", "bce", "ssim", "cosine"]
fig, ax = plt.subplots(figsize=(9, 4.5))
n_m = len(ACTIVE_MODELS)
bar_w = 0.8 / n_m
for mi, m in enumerate(ACTIVE_MODELS):
    means = [df_img[f"{m}_{k}"].mean() for k in IMG_METRICS]
    stds  = [df_img[f"{m}_{k}"].std() for k in IMG_METRICS]
    ax.bar(np.arange(4) + (mi - (n_m - 1) / 2) * bar_w, means, bar_w, yerr=stds,
           capsize=3, color=MODEL_COLORS[MODEL_LABEL[m]], label=MODEL_LABEL[m])
ax.set_xticks(range(4)); ax.set_xticklabels(["MSE", "BCE", "SSIM", "Cosine"])
ax.set_ylabel("Score (mean ± std)")
ax.set_title(f"Image metrics over the complete test split "
             f"(N = {n_eval:,}, best of K={K} samples)")
ax.legend()
save_figure(fig, os.path.join(FIG_DIR, "comparison_image_metrics_overall"))
plt.show()

**Figure 2** — image metrics per magnetic structure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, k in zip(axes.flat, IMG_METRICS):
    for mi, m in enumerate(ACTIVE_MODELS):
        means = [df_img[df_img["structure"] == s][f"{m}_{k}"].mean() for s in STRUCTURE_NAMES]
        stds  = [df_img[df_img["structure"] == s][f"{m}_{k}"].std() for s in STRUCTURE_NAMES]
        ax.bar(np.arange(N_STRUCT) + (mi - (n_m - 1) / 2) * bar_w, means, bar_w, yerr=stds,
               capsize=2, color=MODEL_COLORS[MODEL_LABEL[m]], label=MODEL_LABEL[m])
    ax.set_xticks(range(N_STRUCT))
    ax.set_xticklabels([s.replace(" & ", "\n& ").replace("-", "-\n") for s in STRUCTURE_NAMES],
                       fontsize=7)
    ax.set_title(k.upper())
    ax.set_ylabel("Score (mean ± std)")
axes[0, 0].legend()
fig.suptitle("Image metrics per magnetic structure (6 phases)")
save_figure(fig, os.path.join(FIG_DIR, "comparison_image_metrics_per_structure"))
plt.show()

**Figure 3** — physical metrics overall (Regime B bars, Regime A lighter overlay,
original values as dashed reference lines).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
phys_panels = [("M", "Magnetization M", "orig_M"),
               ("Cnn", "Nearest-neighbor correlation C$_{nn}$", "orig_Cnn"),
               ("chi", "Susceptibility χ", "orig_chi_proxy")]
for ax, (key, title, orig_col) in zip(axes, phys_panels):
    for mi, m in enumerate(ACTIVE_MODELS):
        col_B = f"{m}_chi_ens_B" if key == "chi" else f"{m}_{key}_B"
        col_A = f"{m}_chi_proxy_A" if key == "chi" else f"{m}_{key}_A"
        xpos = mi
        ax.bar(xpos - 0.18, np.nanmean(df_phys[col_B]), 0.34,
               yerr=np.nanstd(df_phys[col_B]), capsize=3,
               color=MODEL_COLORS[MODEL_LABEL[m]], label="Regime B (ensemble)" if mi == 0 else None)
        ax.bar(xpos + 0.18, np.nanmean(df_phys[col_A]), 0.34,
               yerr=np.nanstd(df_phys[col_A]), capsize=3,
               color=MODEL_COLORS[MODEL_LABEL[m]], alpha=0.45,
               label="Regime A (best image)" if mi == 0 else None)
    ax.axhline(np.nanmean(df_phys[orig_col]), ls="--", color="black", lw=1.2,
               label="Original (mean)")
    ax.set_xticks(range(n_m)); ax.set_xticklabels([MODEL_LABEL[m] for m in ACTIVE_MODELS],
                                                  fontsize=8)
    ax.set_title(title)
    ax.legend(fontsize=7)
fig.suptitle("Physical metrics: ensemble (Regime B) vs best image (Regime A)")
save_figure(fig, os.path.join(FIG_DIR, "comparison_physical_metrics_overall"))
plt.show()

**Figure 4** — physical metrics per structure (Regime B).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, (key, title, orig_col) in zip(axes, phys_panels):
    for mi, m in enumerate(ACTIVE_MODELS):
        col_B = f"{m}_chi_ens_B" if key == "chi" else f"{m}_{key}_B"
        means = [np.nanmean(df_phys[df_phys["structure"] == s][col_B]) for s in STRUCTURE_NAMES]
        stds  = [np.nanstd(df_phys[df_phys["structure"] == s][col_B]) for s in STRUCTURE_NAMES]
        ax.bar(np.arange(N_STRUCT) + (mi - (n_m - 1) / 2) * bar_w, means, bar_w, yerr=stds,
               capsize=2, color=MODEL_COLORS[MODEL_LABEL[m]], label=MODEL_LABEL[m])
    orig_means = [np.nanmean(df_phys[df_phys["structure"] == s][orig_col])
                  for s in STRUCTURE_NAMES]
    ax.plot(range(N_STRUCT), orig_means, "k--x", lw=1.2, label="Original")
    ax.set_xticks(range(N_STRUCT))
    ax.set_xticklabels([s.replace(" & ", "\n& ").replace("-", "-\n") for s in STRUCTURE_NAMES],
                       fontsize=7)
    ax.set_title(title)
axes[0].legend(fontsize=7)
fig.suptitle("Physical metrics per magnetic structure (Regime B, K=32 ensembles)")
save_figure(fig, os.path.join(FIG_DIR, "comparison_physical_metrics_per_structure"))
plt.show()

**Figure 5** — the three χ comparisons:
① original proxy vs generated proxy, ② generated proxy vs generated ensemble,
③ original proxy vs generated ensemble. Log-log scatter with the ideal diagonal,
colored by structure; Pearson r² (on log values of valid pairs) in each panel.

In [ ]:
CHI_COMPS = [("chi_orig_proxy", "chi_gen_proxy", "① χ orig (OZ) vs χ gen (OZ)"),
             ("chi_gen_proxy", "chi_gen_ens", "② χ gen (OZ) vs χ gen (FDT ensemble)"),
             ("chi_orig_proxy", "chi_gen_ens", "③ χ orig (OZ) vs χ gen (FDT ensemble)")]

fig, axes = plt.subplots(len(ACTIVE_MODELS), 3,
                         figsize=(13, 4.2 * len(ACTIVE_MODELS)), squeeze=False)
for mi, m in enumerate(ACTIVE_MODELS):
    for ci, (cx, cy, title) in enumerate(CHI_COMPS):
        ax = axes[mi, ci]
        x, y = df_phys[f"{m}_{cx}"].values, df_phys[f"{m}_{cy}"].values
        valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
        for s in STRUCTURE_NAMES:
            sm = valid & (ref_struct == s)
            ax.scatter(x[sm], y[sm], s=6, alpha=0.35, color=STRUCTURE_COLORS[s],
                       label=s, rasterized=True)
        if valid.sum() > 2:
            r, _ = pearsonr(np.log10(x[valid]), np.log10(y[valid]))
            lims = [np.nanmin(x[valid]), np.nanmax(x[valid])]
            ax.plot(lims, lims, "k--", lw=1, label=f"ideal (r²={r**2:.2f})")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel(cx.replace("_", " ")); ax.set_ylabel(cy.replace("_", " "))
        ax.set_title(f"{MODEL_LABEL[m]} — {title}", fontsize=9)
        if mi == 0 and ci == 0:
            ax.legend(fontsize=6)
fig.suptitle("Susceptibility consistency: OZ proxy vs fluctuation-dissipation ensemble")
save_figure(fig, os.path.join(FIG_DIR, "comparison_chi_three_comparisons"))
plt.show()

**Figure 6** — correlation length ξ per structure per model (box plots; only OZ fits
with R² ≥ 0.7 are included, the number of invalid fits is annotated).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))
positions, box_data, box_colors, xticklabels = [], [], [], []
pos = 0
for s in STRUCTURE_NAMES:
    sm = ref_struct == s
    # Reference distribution first, then each model
    series = [("Original", df_phys.loc[sm, "orig_xi"], df_phys.loc[sm, "orig_oz_r2"], "#777777")]
    series += [(MODEL_LABEL[m], df_phys.loc[sm, f"{m}_xi_A"], df_phys.loc[sm, f"{m}_oz_r2_A"],
                MODEL_COLORS[MODEL_LABEL[m]]) for m in ACTIVE_MODELS]
    for name, xi, r2, color in series:
        valid = np.isfinite(xi) & np.isfinite(r2) & (r2 >= OZ_R2_VALID)
        vals = xi[valid].values
        positions.append(pos); box_data.append(vals if len(vals) else [np.nan])
        box_colors.append(color)
        n_inv = int((~valid).sum())
        ax.annotate(f"{n_inv}✗", (pos, 0), xytext=(0, -28), textcoords="offset points",
                    ha="center", fontsize=6, color="#990000")
        pos += 1
    xticklabels.append(s)
    pos += 1
bp = ax.boxplot(box_data, positions=positions, widths=0.7, patch_artist=True,
                showfliers=False)
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
group_w = len(ACTIVE_MODELS) + 1
ax.set_xticks([i * (group_w + 1) + (group_w - 1) / 2 for i in range(N_STRUCT)])
ax.set_xticklabels([s.replace(" & ", "\n& ").replace("-", "-\n") for s in xticklabels],
                   fontsize=8)
ax.set_ylabel("Correlation length ξ (pixels)")
ax.set_title(f"OZ correlation length per structure — only fits with R² ≥ {OZ_R2_VALID}; "
             "✗ = excluded invalid fits")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor="#777777", label="Original")] +
                  [Patch(facecolor=MODEL_COLORS[MODEL_LABEL[m]], label=MODEL_LABEL[m])
                   for m in ACTIVE_MODELS], fontsize=8)
save_figure(fig, os.path.join(FIG_DIR, "comparison_xi_per_structure"))
plt.show()

**Figure 7** — qualitative grid: per structure, the median-SSIM example with the
reference, each model's best sample, and the DDPM difference map.

In [ ]:
n_cols = 2 + len(ACTIVE_MODELS)
fig, axes = plt.subplots(N_STRUCT, n_cols, figsize=(2.4 * n_cols, 2.5 * N_STRUCT),
                         squeeze=False)
for r, s in enumerate(STRUCTURE_NAMES):
    sm = np.where(ref_struct == s)[0]
    ssim_col = f"{ACTIVE_MODELS[0]}_ssim"
    i = sm[np.argsort(df_img.loc[sm, ssim_col].values)[len(sm) // 2]]   # median-SSIM sample
    axes[r, 0].imshow(ref_imgs[i], cmap="jet", vmin=-1, vmax=1)
    axes[r, 0].set_ylabel(s.replace(" & ", "\n& "), fontsize=7)
    for mi, m in enumerate(ACTIVE_MODELS):
        ax = axes[r, mi + 1]
        ax.imshow(best_samples[m][i], cmap="jet", vmin=-1, vmax=1)
        ax.set_xlabel(f"SSIM={df_img.loc[i, f'{m}_ssim']:.3f}\n"
                      f"Cos={df_img.loc[i, f'{m}_cosine']:.3f}", fontsize=7)
    diff = np.abs(ref_imgs[i] - best_samples[ACTIVE_MODELS[0]][i])
    axes[r, n_cols - 1].imshow(diff, cmap="hot", vmin=0, vmax=1)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
for j, t in enumerate(["Reference"] + [MODEL_LABEL[m] for m in ACTIVE_MODELS]
                      + [f"|ref − {MODEL_LABEL[ACTIVE_MODELS[0]]}|"]):
    axes[0, j].set_title(t, fontsize=9)
fig.suptitle("Qualitative comparison (median-SSIM example per structure, 6 phases)")
save_figure(fig, os.path.join(FIG_DIR, "comparison_qualitative_grid"))
plt.show()